# LogCopilot: Production-Oriented Agent Notebook

Этот ноутбук собирает **понятного и управляемого агента** для работы с уже обработанными логами (`run_id`).

Граф шага агента в этом ноутбуке:

1. `input_guard` — проверяем вход.
2. `resolve_context` — поднимаем контекст run из SQLite.
3. `plan` — выбираем action (LLM planner + fallback).
4. `plan_guard` — валидируем/ограничиваем план.
5. `execute_facts` — читаем только разрешенные факты из storage.
6. `bounded_replan` — один безопасный переплан при пустых фактах.
7. `build_visuals` — строим график, если есть готовность.
8. `answer` — LLM объясняет факты человеческим языком.
9. `output_guard` — пост-проверка ответа.
10. `memory_write` — сохраняем память сессии в SQLite.

Важно: агент не читает сырой `.log`, только обработанные данные в БД и зарегистрированные артефакты.

In [ ]:
from __future__ import annotations

import json
import re
import sqlite3
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage

from logcopilot.agent.agent import build_prompt_payload
from logcopilot.agent.config import DEFAULT_DB_PATH, build_chat_model, resolve_model_config
from logcopilot.agent.facts import build_fact_catalog, execute_fact_queries, resolve_run_context
from logcopilot.agent.planner import plan_intent
from logcopilot.agent.prompts import build_answer_system_prompt
from logcopilot.agent.session import AgentSessionState
from logcopilot.agent.visuals import build_visuals_for_plan
from openai.types.responses import tool

In [ ]:
# Базовые ограничения и регулярки для guardrails.
RUN_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{3,128}$")
CLUSTER_ID_RE = re.compile(r"^[a-fA-F0-9]{40}$")
MAX_LIMIT = 20


@dataclass
class GuardResult:
    ok: bool
    reason: str | None = None


@dataclass
class TurnResult:
    answer: str
    profile: str
    plan: dict
    facts: dict
    memory: dict
    trace: list[str] = field(default_factory=list)
    visuals: list[dict] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)


def _extract_text_content(content: Any) -> str:
    """Универсально извлекаем текст из ответа LLM для разных провайдеров."""
    if content is None:
        return ""


    if isinstance(content, list):
        return "".join(_extract_text_content(item) for item in content)
    if isinstance(content, dict):
        for key in ("text", "content", "output_text"):
            value = content.get(key)
            if value:
                return _extract_text_content(value)
        return ""
    return str(content)


def _safe_json(data: Any) -> str:
    return json.dumps(data, ensure_ascii=False, indent=2, default=str)


def input_guard(run_id: str, question: str) -> GuardResult:
    """Проверяем входные параметры до любых запросов в БД/LLM."""
    if not isinstance(run_id, str) or not RUN_ID_RE.match(run_id.strip()):
        return GuardResult(False, "Некорректный run_id: ожидается безопасный идентификатор.")
    if not isinstance(question, str) or not question.strip():
        return GuardResult(False, "Вопрос пустой.")
    if len(question) > 4000:
        return GuardResult(False, "Вопрос слишком длинный (максимум 4000 символов).")
    return GuardResult(True)


def output_guard(answer: str, warnings: list[str]) -> str:
    """Пост-проверка ответа: не оставляем пустой ответ и убираем небезопасные формулировки."""
    text = (answer or "").strip()
    if not text:
        warnings.append("empty_answer_replaced")
        return "Не удалось сформировать ответ по данным run_id. Попробуйте переформулировать вопрос."

    lowered = text.lower()
    if "сырые логи" in lowered or "raw log" in lowered:
        warnings.append("raw_log_claim_detected")
        text += "\n\n[Guard] Ответ построен только по уже обработанным данным run_id."
    return text

In [ ]:
class SQLiteSessionMemoryStore:
    """Простое persistent-хранилище памяти сессии прямо в том же SQLite."""

    def __init__(self, db_path: str = DEFAULT_DB_PATH) -> None:
        self.db_path = Path(db_path)
        self.db_path.parent.mkdir(parents=True, exist_ok=True)
        self._init_schema()

    def _connect(self) -> sqlite3.Connection:
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        return conn

    def _init_schema(self) -> None:
        with self._connect() as conn:
            conn.execute(
                """
                CREATE TABLE IF NOT EXISTS agent_sessions (
                    session_id TEXT NOT NULL,
                    run_id TEXT NOT NULL,
                    memory_json TEXT NOT NULL,
                    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
                    PRIMARY KEY(session_id, run_id)
                )
                """
            )

    def load(self, session_id: str, run_id: str) -> dict:
        with self._connect() as conn:
            row = conn.execute(
                "SELECT memory_json FROM agent_sessions WHERE session_id = ? AND run_id = ?",
                (session_id, run_id),
            ).fetchone()
        if row is None:
            return {}
        try:
            return json.loads(row["memory_json"] or "{}")
        except json.JSONDecodeError:
            return {}

    def save(self, session_id: str, run_id: str, memory: dict) -> None:
        payload = json.dumps(memory or {}, ensure_ascii=False)
        with self._connect() as conn:
            conn.execute(
                """
                INSERT INTO agent_sessions (session_id, run_id, memory_json, updated_at)
                VALUES (?, ?, ?, CURRENT_TIMESTAMP)
                ON CONFLICT(session_id, run_id)
                DO UPDATE SET memory_json = excluded.memory_json, updated_at = CURRENT_TIMESTAMP
                """,
                (session_id, run_id, payload),
            )

In [21]:
impo
@tool
def add(a : int, b :int) -> str:
    """Складывает два целых числа a и b."""
    if not a:
        return json.dumps({
            'status' : 'error',
            'message' : f'Func call without attr: {a}'
        })
    if not b:
        return json.dumps({
            'status' : 'error',
            'message' : f'Func call without attr: {b}'
        })
    return json.dumps({
        'status' : 'success',
        'message' : f'{a} + {b} = {a + b}'
    })

NameError: name 'tool' is not defined

In [19]:
import os
from dotenv import load_dotenv
from langchain_community.chat_models import ChatYandexGPT
from langchain_core.messages import HumanMessage

load_dotenv()

# Инициализация родного класса YandexGPT
llm = ChatYandexGPT(
    api_key=os.getenv('YC_AI_API_KEY'),
    folder_id=os.getenv('YC_FOLDER_ID'),
    model_uri=f"gpt://{os.getenv('YC_FOLDER_ID')}/yandexgpt/latest",
)

Привет! Всё хорошо, спасибо. А у вас как дела?


In [ ]:
try:
    llm.bind_tools([add])
except Exception as e:
    print(f'Ошибка: {e}')

In [ ]:
def _sanitize_plan(plan: dict, fact_catalog: dict, memory_payload: dict) -> dict:
    """Жестко валидируем plan, чтобы planner не вышел за контракт."""
    allowed_actions = set(fact_catalog.get("available_actions", []))
    safe_action = plan.get("action", "overview")
    if safe_action not in allowed_actions:
        safe_action = "overview"

    args = dict(plan.get("args") or {})
    visual_hint = plan.get("visual_hint") if isinstance(plan.get("visual_hint"), dict) else None

    if "limit" in args:
        try:
            args["limit"] = max(1, min(int(args["limit"]), MAX_LIMIT))
        except Exception:
            args["limit"] = 5

    # Для incident_cluster_* обязателен cluster_id в корректном формате.
    if safe_action in {"incident_cluster_detail", "incident_cluster_examples", "incident_cluster_causes"}:
        cluster_id = str(args.get("cluster_id") or "")
        if not CLUSTER_ID_RE.match(cluster_id):
            memory = AgentSessionState.from_dict(memory_payload)
            if memory.focused_cluster_id and CLUSTER_ID_RE.match(memory.focused_cluster_id):
                args["cluster_id"] = memory.focused_cluster_id
            else:
                safe_action = "top_incidents"
                args = {"limit": 3}
                visual_hint = {"kind": "incidents_top_clusters"}

    return {
        "action": safe_action,
        "args": args,
        "visual_hint": visual_hint,
    }


def _remember(plan: dict, facts: dict, memory_payload: dict | None, profile: str) -> dict:
    """Минимальная память для follow-up вопросов."""
    memory = AgentSessionState.from_dict(memory_payload)
    action = plan.get("action")

    memory.turn_index += 1
    memory.last_action = action
    memory.last_profile_scope = profile

    if action == "top_incidents":
        rows = facts.get("top_incidents", [])
        cluster_ids = [row.get("cluster_id") for row in rows if isinstance(row, dict) and row.get("cluster_id")]
        if cluster_ids:
            memory.last_ranked_cluster_ids = cluster_ids
            memory.last_top_cluster_ids = cluster_ids
            memory.focused_cluster_id = cluster_ids[0]
            memory.last_topic = "incident_ranking"

    if action in {"incident_cluster_detail", "incident_cluster_examples", "incident_cluster_causes"}:
        row = facts.get("incident_cluster")
        if isinstance(row, dict) and row.get("cluster_id"):
            memory.focused_cluster_id = row["cluster_id"]
            memory.last_topic = "incident_cluster"

    if action in {"parser_diagnostics_summary", "profile_fit_summary"}:
        memory.last_topic = action

    return memory.to_dict()

In [ ]:
def _fallback_answer(plan: dict, facts: dict) -> str:
    """Fallback-ответ, если LLM недоступна."""
    action = plan.get("action")

    if action == "capabilities":
        examples = facts.get("examples", [])
        examples_text = "\n".join(f"- {x}" for x in examples)
        return f"{facts.get('message', 'Я готов помочь.')}\n{examples_text}".strip()

    if action == "profile_scope":
        return facts.get("message", "Запрошенный сценарий недоступен для этого run_id.")

    if action == "top_incidents":
        rows = facts.get("top_incidents", [])
        if not rows:
            return "Инциденты не найдены."
        top = rows[0]
        return (
            f"Главный инцидент: {top.get('cluster_id')} "
            f"(повторов: {top.get('incident_hits', top.get('hits', 0))})."
        )

    if action in {"incident_cluster_detail", "incident_cluster_examples", "incident_cluster_causes"}:
        row = facts.get("incident_cluster") or {}
        if not row:
            return "Кластер не найден."
        return (
            f"Кластер {row.get('cluster_id')}: "
            f"повторов {row.get('incident_hits', row.get('hits', 0))}. "
            f"Суть: {row.get('representative_text', 'n/a')}"
        )

    if action == "heatmap":
        rows = facts.get("heatmap", [])
        if not rows:
            return "Heatmap-данные отсутствуют."
        top = rows[0]
        return (
            f"Пик: {top.get('bucket_start')} | {top.get('component')} | "
            f"{top.get('operation')} | hits={top.get('hits')}"
        )

    if action in {"traffic_summary", "traffic_500"}:
        rows = facts.get("traffic_summary", [])
        if not rows:
            return "Traffic-данные отсутствуют."
        top = rows[0]
        return (
            f"Топ endpoint: {top.get('method')} {top.get('path')} "
            f"status={top.get('http_status')} hits={top.get('hits')}"
        )

    if action == "traffic_anomalies":
        rows = facts.get("traffic_anomalies", [])
        if not rows:
            return "Аномалии трафика не найдены."
        first = rows[0]
        return f"Аномалия: [{first.get('severity')}] {first.get('title')}"

    profile = facts.get("profile", "unknown")
    events = facts.get("run_summary", {}).get("event_count", "n/a")
    return f"Профиль: {profile}. Обработано событий: {events}."


def run_turn(
    question: str,
    run_id: str,
    session_id: str = "default",
    provider: str = "yandex",
    db_path: str = DEFAULT_DB_PATH,
    use_llm_planner: bool = True,
    model: str | None = None,
    base_url: str | None = None,
    api_key: str | None = None,
    folder_id: str | None = None,
) -> TurnResult:
    """
    Один полный шаг агента (один user turn) с guardrails, памятью и bounded replan.
    """
    trace: list[str] = []
    warnings: list[str] = []

    # 1) Input guard
    guard = input_guard(run_id=run_id, question=question)
    if not guard.ok:
        return TurnResult(
            answer=guard.reason or "Ошибка входных данных.",
            profile="unknown",
            plan={"action": "guard_block"},
            facts={},
            memory={},
            trace=["input_guard: blocked"],
            visuals=[],
            warnings=[],
        )
    trace.append("input_guard: ok")

    # 2) Resolve context
    context = resolve_run_context(run_id=run_id, db_path=db_path)
    trace.append(f"resolve_context: profile={context.profile}")

    # 3) Поднимаем память текущей сессии
    memory_store = SQLiteSessionMemoryStore(db_path=db_path)
    memory_payload = memory_store.load(session_id=session_id, run_id=run_id)
    trace.append(f"memory_read: turn_index={memory_payload.get('turn_index', 0)}")

    # 4) Поднимаем LLM (planner + answer). Если недоступна, продолжаем с fallback.
    llm = None
    if provider and provider != "none":
        try:
            llm = ChatOpenAI(
                provider = 'yandex',
                model = f"gpt://{os.getenv('YC_FOLDER_ID')}/yandexgpt/yandexgpt-5.1",
                api_key = os.getenv('YC_AI_API_KEY'),
                folder_id = os.getenv('YC_FOLDER_ID')
            )
            trace.append(f"llm_init: provider={provider}")
        except Exception as exc:
            warnings.append(f"llm_unavailable: {exc}")
            trace.append("llm_init: failed -> fallback")

    # 5) Plan + guard
    fact_catalog = build_fact_catalog(context)
    planner_model = llm if (use_llm_planner and llm is not None) else None
    raw_plan = plan_intent(
        context=context,
        question=question,
        fact_catalog=fact_catalog,
        memory_payload=memory_payload,
        planner_model=planner_model,
    )
    plan = _sanitize_plan(raw_plan, fact_catalog=fact_catalog, memory_payload=memory_payload)
    trace.append(f"plan: action={plan.get('action')} args={plan.get('args', {})}")

    # 6) Execute facts
    facts = execute_fact_queries(context, plan)
    trace.append(f"execute_facts: keys={sorted(facts.keys())}")

    # 7) Bounded replan: максимум одна попытка, если action выбрался, но факты пустые.
    expected_fact_key = {
        "top_incidents": "top_incidents",
        "incident_cluster_detail": "incident_cluster",
        "incident_cluster_examples": "incident_cluster",
        "incident_cluster_causes": "incident_cluster",
        "heatmap": "heatmap",
        "traffic_summary": "traffic_summary",
        "traffic_500": "traffic_summary",
        "traffic_anomalies": "traffic_anomalies",
    }.get(plan.get("action"))

    if expected_fact_key and not facts.get(expected_fact_key):
        warnings.append("bounded_replan_triggered")
        fallback_plan = {"action": "overview", "args": {}, "visual_hint": None}
        facts = execute_fact_queries(context, fallback_plan)
        plan = fallback_plan
        trace.append("bounded_replan: switched_to_overview")

    # 8) Build visuals
    visuals, visual_info = build_visuals_for_plan(context, plan, facts)
    facts = {**facts, "visual_info": visual_info}
    trace.append(f"build_visuals: status={visual_info.get('status')}")

    # 9) Generate answer
    prompt_payload = build_prompt_payload(plan=plan, facts=facts, visuals=visuals)
    messages = [
        SystemMessage(content=build_answer_system_prompt(context.profile)),
        HumanMessage(
            content=(
                f"Вопрос пользователя:\n{question}\n\n"
                f"Выбранное действие:\n{_safe_json(plan)}\n\n"
                f"Компактные факты из БД:\n{_safe_json(prompt_payload)}\n\n"
                "Ответь кратко и по делу."
            )
        ),
    ]

    if llm is not None:
        try:
            response = llm.invoke(messages)
            answer = _extract_text_content(getattr(response, "content", response))
            trace.append("answer: llm_invoke")
        except Exception as exc:
            warnings.append(f"answer_llm_failed: {exc}")
            answer = _fallback_answer(plan, facts)
            trace.append("answer: fallback_after_llm_error")
    else:
        answer = _fallback_answer(plan, facts)
        trace.append("answer: fallback_no_llm")

    # 10) Output guard
    answer = output_guard(answer, warnings)
    trace.append("output_guard: ok")

    # 11) Memory write
    memory = _remember(plan=plan, facts=facts, memory_payload=memory_payload, profile=context.profile)
    memory_store.save(session_id=session_id, run_id=run_id, memory=memory)
    trace.append(f"memory_write: focused={memory.get('focused_cluster_id') or 'none'}")

    return TurnResult(
        answer=answer,
        profile=context.profile,
        plan=plan,
        facts=facts,
        memory=memory,
        trace=trace,
        visuals=visuals,
        warnings=warnings,
    )

In [ ]:
def evaluate_router(
    cases: list[dict],
    run_id: str,
    db_path: str = DEFAULT_DB_PATH,
    provider: str = "none",
) -> dict:
    """
    Мини-harness для benchmark маршрутизации.
    cases: [{"question": "...", "expected_action": "top_incidents"}, ...]
    """
    if not cases:
        return {"total": 0, "accuracy": 0.0, "rows": []}

    rows = []
    correct = 0
    for idx, case in enumerate(cases, start=1):
        question = case["question"]
        expected = case["expected_action"]
        session_id = f"eval_{idx}"

        result = run_turn(
            question=question,
            run_id=run_id,
            session_id=session_id,
            provider=provider,
            db_path=db_path,
            use_llm_planner=False,
        )

        predicted = result.plan.get("action")
        is_ok = predicted == expected
        correct += int(is_ok)
        rows.append(
            {
                "question": question,
                "expected_action": expected,
                "predicted_action": predicted,
                "ok": is_ok,
            }
        )

    return {
        "total": len(cases),
        "correct": correct,
        "accuracy": round(correct / len(cases), 3),
        "rows": rows,
    }

## Как использовать

1. Убедитесь, что у вас уже есть обработанный `run_id` в `out/logcopilot.sqlite`.
2. В `.env` должны быть ключи для провайдера (`YC_FOLDER_ID`, `YC_AI_API_KEY` или local).
3. Не храните API-ключи прямо в ноутбуке.

Пример одного шага:

```python
result = run_turn(
    question="Покажи топ-3 инцидента и объясни, что сломалось",
    run_id="<ВАШ_RUN_ID>",
    session_id="user_42",
    provider="yandex",  # или local/none
)

print(result.answer)
print(result.plan)
print(result.trace)
print(result.visuals)
```

Пример простого benchmark маршрутизации:

```python
cases = [
    {"question": "покажи топ инциденты", "expected_action": "top_incidents"},
    {"question": "почему качество разбора низкое", "expected_action": "parser_diagnostics_summary"},
]
report = evaluate_router(cases, run_id="<ВАШ_RUN_ID>")
print(report["accuracy"], report["rows"])
```